# Manual estimation of DML

## Estimators


In [20]:
import numpy as np
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from scipy.stats import norm


def estimate_dml_manual(
    X,
    D,
    Y,
    tau_true=None,
    f_x=None,
    e_true=None,
    n_splits=5,
    random_state=42
):
    X = np.asarray(X)
    D = np.asarray(D).ravel().astype(float)
    Y = np.asarray(Y).ravel().astype(float)

    n = len(Y)

    model_y = RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=5,
        random_state=random_state,
        n_jobs=1
    )
    model_t = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=5,
        random_state=random_state,
        n_jobs=1
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    m_hat = np.zeros(n)
    e_hat = np.zeros(n)

    for train_idx, test_idx in cv.split(X, D):
        X_tr, X_te = X[train_idx], X[test_idx]
        D_tr = D[train_idx]
        Y_tr = Y[train_idx]

        my = clone(model_y)
        mt = clone(model_t)

        my.fit(X_tr, Y_tr)
        mt.fit(X_tr, D_tr)

        m_hat[test_idx] = my.predict(X_te)
        e_hat[test_idx] = mt.predict_proba(X_te)[:, 1]

    Y_res = Y - m_hat
    D_res = D - e_hat

    # Final partialling-out regression without intercept
    reg = LinearRegression(fit_intercept=False)
    reg.fit(D_res.reshape(-1, 1), Y_res)
    tau_hat = float(reg.coef_[0])

    # Simple HC0-style variance estimate
    u = Y_res - tau_hat * D_res
    denom = np.mean(D_res ** 2) ** 2
    var_hat = np.mean((D_res ** 2) * (u ** 2)) / (n * denom)
    se = float(np.sqrt(var_hat))

    z = norm.ppf(0.975)
    ci_lower = tau_hat - z * se
    ci_upper = tau_hat + z * se

    out = {
        "tau_hat": tau_hat,
        "se": se,
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "m_hat_oof": m_hat,
        "e_hat_oof": e_hat,
        "resid_d_var": np.var(D_res),
        "resid_d_sq_mean": float(np.mean(D_res**2))
    }

    if tau_true is not None:
        out["tau_bias"] = float(tau_hat - tau_true)

    if f_x is not None and e_true is not None:
        f_x = np.asarray(f_x).ravel()
        e_true = np.asarray(e_true).ravel()
        m_true = f_x + (tau_true * e_true if tau_true is not None else 0.0)

        out["m_mse"] = float(mean_squared_error(m_true, m_hat))
        out["e_mse"] = float(mean_squared_error(e_true, e_hat))
        out["m_rmse"] = float(np.sqrt(out["m_mse"]))
        out["e_rmse"] = float(np.sqrt(out["e_mse"]))

    return out

In [21]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.estimator import estimate_ols
from src.dgp import generate_dataset
from src.utils import load_config

import pandas as pd
import numpy as np
import pyarrow
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool
import os


def one_replication(config, alpha_y, alpha_d, kappa, seed, replication):
    data = generate_dataset(config, alpha_y=alpha_y, alpha_d=alpha_d, kappa=kappa, seed=seed)

    ols_res = estimate_ols(data["X"], data["D"], data["Y"])

    dml_res = estimate_dml_manual(data["X"], data["D"], data["Y"], data["tau_true"], data["f_alpha"], data["e"])

    return {
        "alpha_y": alpha_y,
        "alpha_d": alpha_d,
        "kappa": kappa,
        "seed": seed,
        "overlap": np.mean(data["e"] * (1 - data["e"])),
        "residual_d_var": np.var(data["D"] - data["e"]),
        "replication": replication,
        "tau_true": data["tau_true"],
        "ols_tau_hat": ols_res["tau_hat"],
        "ols_se": ols_res["se"],
        "ols_ci_lower": ols_res["ci_lower"],
        "ols_ci_upper": ols_res["ci_upper"],
        "dml_tau_hat": dml_res["tau_hat"],
        "dml_se": dml_res["se"],
        "dml_ci_lower": dml_res["ci_lower"],
        "dml_ci_upper": dml_res["ci_upper"],
        "m_mse": dml_res["m_rmse"],
        "e_mse": dml_res["e_rmse"],
        "estimated_resid_var": dml_res["resid_d_var"], 
        "treatmeant_var": dml_res["resid_d_sq_mean"]
    }


def run_scenario(
    config: dict,
    alpha_y: float,
    alpha_d: float,
    kappa: float,
    n_jobs: int | None = None,
) -> pd.DataFrame:
    base_seed = config["random_seed"]
    R = config["num_replications"]
    cpu_total = os.cpu_count() or 1
    n_jobs = max(1, min(int(n_jobs or cpu_total), cpu_total))

    def _run_serial() -> list[dict]:
        serial_results = []
        for r in tqdm(
            range(R),
            total=R,
            desc=f"alpha_y={alpha_y}, alpha_d={alpha_d}, kappa={kappa} (serial)",
        ):
            seed = base_seed + r
            serial_results.append(
                one_replication(
                    config,
                    alpha_y,
                    alpha_d,
                    kappa,
                    seed,
                    r,
                )
            )
        return serial_results

    if n_jobs == 1:
        results = _run_serial()
        return pd.DataFrame(results).sort_values("replication").reset_index(drop=True)

    results = []
    future_to_rep = {}

    try:
        with ProcessPoolExecutor(max_workers=n_jobs) as executor:
            for r in range(R):
                seed = base_seed + r
                future = executor.submit(
                    one_replication,
                    config,
                    alpha_y,
                    alpha_d,
                    kappa,
                    seed,
                    r,
                )
                future_to_rep[future] = r

            for fut in tqdm(
                as_completed(future_to_rep),
                total=R,
                desc=f"alpha_y={alpha_y}, alpha_d={alpha_d}, kappa={kappa}",
            ):
                rep = future_to_rep[fut]
                try:
                    results.append(fut.result())
                except Exception as exc:
                    raise RuntimeError(
                        f"Replication {rep} failed in worker process. "
                        "Re-run with n_jobs=1 to surface the original traceback."
                    ) from exc

    except BrokenProcessPool:
        print(
            "Process pool crashed (BrokenProcessPool). "
            "Falling back to serial execution for this scenario."
        )
        results = _run_serial()

    return pd.DataFrame(results).sort_values("replication").reset_index(drop=True)


def run_simulation_grid(config: dict, save_each: bool = True, n_jobs: int | None = None) -> pd.DataFrame:
    all_results = []

    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    for alpha_y in config["alpha_y_grid"]:
        for alpha_d in config["alpha_d_grid"]:
            for kappa in config["kappa"]:
                scenario_df = run_scenario(
                    config=config,
                    alpha_y=alpha_y,
                    alpha_d=alpha_d,
                    kappa=kappa,
                    n_jobs=n_jobs,
                )
                all_results.append(scenario_df)

                if save_each:
                    filename = output_dir / f"alpha_y_{alpha_y}_alpha_d_{alpha_d}_kappa_{kappa}.csv"
                    scenario_df.to_parquet(filename.with_suffix(".parquet"), index=False)

    return pd.concat(all_results, ignore_index=True)


In [22]:
config = load_config("baseline")


In [23]:
import os
df = run_simulation_grid(config, save_each=False, n_jobs=1)

alpha_y=1, alpha_d=1, kappa=1 (serial): 100%|██████████| 10/10 [00:15<00:00,  1.58s/it]


In [24]:
df

,alpha_y,alpha_d,kappa,seed,overlap,residual_d_var,replication,tau_true,ols_tau_hat,ols_se,ols_ci_lower,ols_ci_upper,dml_tau_hat,dml_se,dml_ci_lower,dml_ci_upper,m_mse,e_mse,estimated_resid_var,treatmeant_var
0,1,0.0,1,42,0.207883,0.218841,0,2.5,2.414286,0.202866,2.016669,2.811904,2.304624,0.177908,1.955932,2.653317,0.976839,0.152414,0.245589,0.245590
1,1,0.0,1,43,0.204363,0.196876,1,2.5,2.482394,0.247787,1.996731,2.968056,2.270819,0.221772,1.836154,2.705485,0.995002,0.133057,0.222607,0.222607
2,1,0.0,1,44,0.208623,0.199314,2,2.5,2.503131,0.245694,2.021570,2.984692,2.368578,0.209213,1.958528,2.778629,1.045485,0.129900,0.226703,0.226720
3,1,0.0,1,45,0.206516,0.218066,3,2.5,2.421708,0.244378,1.942727,2.900688,2.393134,0.206903,1.987611,2.798657,1.084763,0.163506,0.233551,0.233594
4,1,0.0,1,46,0.207007,0.197485,4,2.5,2.201805,0.235756,1.739723,2.663888,2.196478,0.187325,1.829329,2.563628,1.047149,0.119623,0.223373,0.223375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,1,1.0,1,47,0.209389,0.200613,5,2.5,2.858661,0.261125,2.346855,3.370467,2.819895,0.212931,2.402558,3.237231,1.037508,0.158522,0.225175,0.225208
86,1,1.0,1,48,0.205362,0.193638,6,2.5,2.580429,0.348262,1.897837,3.263022,2.598847,0.205381,2.196308,3.001386,0.936476,0.157614,0.224857,0.224871
87,1,1.0,1,49,0.213340,0.203518,7,2.5,2.711532,0.241402,2.238383,3.184680,2.380684,0.199170,1.990318,2.771051,0.832077,0.156729,0.245185,0.245270
88,1,1.0,1,50,0.207658,0.179922,8,2.5,2.083294,0.240972,1.610989,2.555599,2.173679,0.211617,1.758917,2.588441,0.952674,0.149862,0.221724,0.221724
